# Respiratory Sound Classification: Project Summary & Experiments

This notebook serves as a centralized documentation hub for the ICBHI respiratory sound classification project. It tracks the evolution of the pipeline from the baseline Audio Spectrogram Transformer (AST) to the advanced Self-Supervised Audio Spectrogram Transformer (SSAST).

## 🧪 Experiment Progression
- **Baseline**: Audio Spectrogram Transformer (`base_ast.py`)
- **Experiment P1**: Wavelets Integration (`exp_p1_wavelets.yaml`)
- **Experiment P2**: Focal Loss (`exp_p2_focal.yaml`) to handle class imbalance.
- **Experiment P3**: ASAM Optimizer (`exp_p3_asam.yaml`) for smoother loss landscapes.
- **Experiment P4**: SSAST (`exp_p4_ssast.yaml`) — Shifting to a self-supervised pre-trained backbone for robust feature extraction.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

# Add root project to path to allow importing from src
sys.path.append(os.path.abspath('..'))

from src.models.base_ast import ASTClassifier
from src.models.ssast import SSASTModel
from src.data.dataset import BreathDataset, SSASTBreathDataset
from src.utils.metrics import compute_icbhi_metrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Data Pipeline
In P4, we transitioned from using standard `AutoFeatureExtractor` HuggingFace utilities to calculating raw **Log-Mel Spectrograms** on the fly using `torchaudio`. This provides the precise input format required by the SSAST backbone.

In [ ]:
# Example of instantiating the datasets (assumes preprocessed .npz exists)
DATA_PATH = "../data/processed/icbhi_ast_16k_8s.npz"

try:
    data = np.load(DATA_PATH)
    X_test, y_test, d_test = data['X_test'], data['y_test'], data['device_test']
    print(f"Loaded {len(X_test)} samples for evaluation.")
    
    # SSAST Dataset uses torchaudio MelSpectrogram internally
    ssast_dataset = SSASTBreathDataset(X_test, y_test, d_test, train=False)
    test_loader = DataLoader(ssast_dataset, batch_size=8, shuffle=False)
    print("DataLoader successfully initialized!")
except FileNotFoundError:
    print(f"Data file not found at {DATA_PATH}. Please run preprocessing first.")

## 2. Models: Baseline AST vs SSAST
The core of the recent work (P4) involved porting over the SSAST model. Unlike standard AST, SSAST initializes the Vision Transformer with weights pre-trained via self-supervised masking on massive audio datasets, then adds a custom MLP classification head.

In [ ]:
def build_models():
    print("--- Baseline AST ---")
    ast_model = ASTClassifier(num_classes=4)
    print(f"AST Initialized. Parameter count: {sum(p.numel() for p in ast_model.parameters()) / 1e6:.2f}M")
    
    print("\n--- SSAST (Base) ---")
    # Note: Requires the pretrained .pth weights downloaded in /pretrained
    pretrained_path = "../pretrained/SSAST-Base-Patch-400.pth"
    if os.path.exists(pretrained_path):
        ssast_model = SSASTModel(
            label_dim=4, fshape=16, tshape=16, fstride=10, tstride=10,
            input_fdim=128, input_tdim=1024, model_size='base',
            load_pretrained_mdl_path=pretrained_path
        )
        print(f"SSAST Initialized. Parameter count: {sum(p.numel() for p in ssast_model.parameters()) / 1e6:.2f}M")
    else:
        print(f"SSAST pretrained weights not found at {pretrained_path}.")

build_models()

## 3. Evaluation Routine & Hardware Optimization
To accurately evaluate the ICBHI dataset, we use a specialized scoring metric (`compute_icbhi_metrics`) which calculates Sensitivity (Se), Specificity (Sp), and the Harmonic Mean / ICBHI Score.

> **GPU Note**: Evaluating 2,750+ 8-second audio clips with an 86 Million parameter Transformer is extremely computationally heavy. If `torch.cuda.is_available()` is False, this block defaults to CPU and can take upwards of 25 minutes. With a CUDA-enabled GPU (even a mobile GPU like the MX350), evaluation drops to under 1 minute.

In [ ]:
# Example logic (mimicking evaluate.py)
def pseudo_evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    
    print("Starting inference...")
    with torch.no_grad():
        # Simulate running 2 batches just for the notebook
        for i, (inputs, labels, _) in enumerate(dataloader):
            if i >= 2: 
                break 
            
            inputs = inputs.to(DEVICE)
            logits = model(inputs)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Standard ICBHI metric computation
    se, sp, score, cm = compute_icbhi_metrics(all_labels, all_preds)
    
    print(f"\n{'='*40}")
    print(f"  Sensitivity (Se): {se:.4f}")
    print(f"  Specificity (Sp): {sp:.4f}")
    print(f"  ICBHI Score:      {score:.4f}")
    print(f"{'='*40}\n")
    
    # Confusion Matrix Visualization
    labels_list = ['Normal', 'Crackle', 'Wheeze', 'Both']
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_list, yticklabels=labels_list)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix (Score={score:.4f})')
    plt.show()

print("Evaluation routine defined.")